In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import seaborn as sns

### PCA: Principal Component Analysis

**PCA** finds new axes (principal components) that capture the **maximum variance** in your data.

Think of it as:

Original Data (4 features): PCA (2 features):

[sepal_length, sepal_width, petal_length, petal_width] → [PC1, PC2]



**The Goal:** Reduce dimensions while preserving as much information as possible!

#### The Concept
*"Find the directions where the data varies the most"*

| Step | Action |
|------|--------|
| **Step 1** | Find direction of **maximum variance** (PC1) |
| **Step 2** | Find **perpendicular** direction of next maximum variance (PC2) |
| **Step 3** | **Project** data onto these new axes |

**Result:** Data is rotated to maximize variance along new axes.

#### Interpreting Components

| Component | Key Features | Interpretation |
|-----------|--------------|----------------|
| **PC1** | petal_length (+0.89), petal_width (+0.87) | Petal size (both increase together) |
| **PC2** | sepal_length (+0.65), sepal_width (-0.73) | Sepal shape (length vs width trade-off) |
> **First 2 components explain 96.4% of variance!** ✅


PCA reduces dimensions while keeping the most important information!
  
| Before PCA (4 correlated features) | → PCA → | After PCA (2 principal components) |
|-----------------------------------|---------|-------------------------------------|
| sepal_length: ⬤⬤⬤⬤⬤             |         | PC1 (73.6%): ⬤⬤⬤⬤⬤⬤⬤⬤⬤⬤        |
| petal_length: ⬤⬤⬤⬤⬤             |         | PC2 (22.8%): ⬤⬤⬤⬤⬤⬤⬤⬤⬤⬤        |
| sepal_width: ⬤⬤⬤⬤⬤              |         |                                     |
| petal_width: ⬤⬤⬤⬤⬤              |         |                                     |
| ❌ Redundant information          |         | ✅ 96.4% variance preserved!       |

#### ✅ DO

- ALWAYS scale your data before PCA
- Check explained variance to choose number of components
- Use PCA for visualization (reduce to 2D/3D)
- Use PCA for noise reduction
- Use PCA as preprocessing for other algorithms

#### ❌ DON'T

- Use PCA without scaling
- Keep too many components (defeats the purpose)
- Interpret PC directions without looking at loadings
- Use PCA if interpretability is crucial

#### 💡 When to Use PCA

- Visualizing high-dimensional data
- Reducing training time (fewer features → faster)
- Removing noise/overfitting
- Feature engineering for other models

#### 💡 When NOT to Use PCA

- When you need interpretable features
- When you have categorical data
- When you need exact values (not approximations)
- When you have missing data (PCA can't handle it)


#### The Golden Rule

> **PCA is a tool for compression, not interpretation!**

In [7]:
# data
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

data = pd.DataFrame(X, columns=feature_names)
data['species'] = y
data['species_name'] = data['species'].map({i: name for i, name in enumerate(target_names)})

# scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# model
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

# variance
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Explained Variance by Each Component:")
for i, ev in enumerate(explained_variance):
    print(f"  PC{i+1}: {ev*100:.2f}%")

print(f"\nCumulative Explained Variance:")
for i, cv in enumerate(cumulative_variance):
    print(f"  PC{i+1}: {cv*100:.2f}%")

# we can reduce from 4 features to 2 with only 4% information loss
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

data_pca = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
data_pca['species'] = y
data_pca['species_name'] = data_pca['species'].map({i: name for i, name in enumerate(target_names)})
data_pca

Explained Variance by Each Component:
  PC1: 72.96%
  PC2: 22.85%
  PC3: 3.67%
  PC4: 0.52%

Cumulative Explained Variance:
  PC1: 72.96%
  PC2: 95.81%
  PC3: 99.48%
  PC4: 100.00%


,PC1,PC2,species,species_name
0,-2.264703,0.480027,0,setosa
1,-2.080961,-0.674134,0,setosa
2,-2.364229,-0.341908,0,setosa
3,-2.299384,-0.597395,0,setosa
4,-2.389842,0.646835,0,setosa
...,...,...,...,...
145,1.870503,0.386966,2,virginica
146,1.564580,-0.896687,2,virginica
147,1.521170,0.269069,2,virginica
148,1.372788,1.011254,2,virginica


In [16]:
loadings_df = pd.DataFrame(
    pca_2d.components_.T,
    columns=['PC1', 'PC2'],
    index=feature_names
)

print("Feature Loadings (Correlation with PCs):")
print(loadings_df)

print("\nInterpretation:")
print(f"PC1 (73.6% of variance):")
print("  → Strongly influenced by petal features")
print("  → petal_length ({:.3f}) and petal_width ({:.3f})".format(
    loadings_df.loc['petal length (cm)', 'PC1'],
    loadings_df.loc['petal width (cm)', 'PC1']
))
print("  → This separates Setosa from the others!")

print(f"\nPC2 (22.8% of variance):")
print("  → Strongly influenced by sepal features")
print("  → sepal_length ({:.3f}) and sepal_width ({:.3f})".format(
    loadings_df.loc['sepal length (cm)', 'PC2'],
    loadings_df.loc['sepal width (cm)', 'PC2']
))
print("  → This separates Versicolor from Virginica!")

print("\n**Petal Length by Species:")
for i, species in enumerate(target_names):
    petal_length = X[y == i, 2]
    print(f"  {species}: mean={petal_length.mean():.2f}, std={petal_length.std():.2f}")

print("\n" + "-"*60)
print("Setosa has: VERY SHORT petals ({:.2f} cm)".format(X[y==0, 2].mean()))
print("Versicolor has: MEDIUM petals ({:.2f} cm)".format(X[y==1, 2].mean()))
print("Virginica has: LONG petals ({:.2f} cm)".format(X[y==2, 2].mean()))

print("\n" + "="*60)
print("WHY PC1 SEPARATES SETOSA FROM THE OTHERS:")
print("  • PC1 has HIGH loadings for petal features")
print("  • Setosa has SMALL petals → LOW PC1 values")
print("  • Versicolor/Virginica have LARGER petals → HIGH PC1 values")
print("  → PC1 creates a clear gap between Setosa and the others!")

print("\n**Sepal Width by Species:")
for i, species in enumerate(target_names):
    sepal_width = X[y == i, 1]
    print(f"  {species}: mean={sepal_width.mean():.2f}, std={sepal_width.std():.2f}")

print("\n" + "-"*60)
print("Versicolor has: WIDER sepals ({:.2f} cm)".format(X[y==1, 1].mean()))
print("Virginica has: NARROWER sepals ({:.2f} cm)".format(X[y==2, 1].mean()))

print("\n" + "="*60)
print("WHY PC2 SEPARATES VERSICOLOR FROM VIRGINICA:")
print("  • PC2 has HIGH negative loading for sepal_width")
print("  • Versicolor has WIDER sepals → LOWER PC2 values")
print("  • Virginica has NARROWER sepals → HIGHER PC2 values")
print("  → PC2 separates Versicolor from Virginica!")


Feature Loadings (Correlation with PCs):
                        PC1       PC2
sepal length (cm)  0.521066  0.377418
sepal width (cm)  -0.269347  0.923296
petal length (cm)  0.580413  0.024492
petal width (cm)   0.564857  0.066942

Interpretation:
PC1 (73.6% of variance):
  → Strongly influenced by petal features
  → petal_length (0.580) and petal_width (0.565)
  → This separates Setosa from the others!

PC2 (22.8% of variance):
  → Strongly influenced by sepal features
  → sepal_length (0.377) and sepal_width (0.923)
  → This separates Versicolor from Virginica!

**Petal Length by Species:
  setosa: mean=1.46, std=0.17
  versicolor: mean=4.26, std=0.47
  virginica: mean=5.55, std=0.55

------------------------------------------------------------
Setosa has: VERY SHORT petals (1.46 cm)
Versicolor has: MEDIUM petals (4.26 cm)
Virginica has: LONG petals (5.55 cm)

WHY PC1 SEPARATES SETOSA FROM THE OTHERS:
  • PC1 has HIGH loadings for petal features
  • Setosa has SMALL petals → LOW PC1 